# skill

> one vault for everything you have read: web pages, papers, video, files, code and your own notes in one SQLite
> file, searchable together and answerable by a local or hosted model

In [ ]:
#| default_exp skill

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

#| export
## Getting started

```python
from vishalakshi.skill import vault
v = vault()                      # $VISHALAKSHI_VAULT, else ~/.vishalakshi/vault.db
v.add('~/notes')                 # directory, file, or text
v.grab('1706.03762')             # arXiv id, YouTube url, GitHub repo, PDF url, path
v.note('what I concluded', tags=['retrieval'])
```

## Reading it back

```python
v.context(q)                     # whole sections plus what they connect to; hand this to a model
v.ask(q)                         # answer with [n] citations -> node_ids
v.search(q, limit=10)            # chunk hits
v.sections(q, limit=5)           # ranked sections
v.read(node_id)                  # section behind a citation
```

`kind` filters without a second pass: `v.search(q, kind='pdf')` or `'note,web'`.
Kinds: `web`, `pdf`, `arxiv`, `youtube`, `file`, `code`, `data`, `note`.

## Named documents

```python
v.ask_doc(['a.py', 'b.py'], 'what does a do that b does not?')
v.ask_doc(path, 'what is owed and by when?', schema='amount:float, due:str')
```

A path on disk is read even if not yet in the vault. `doc_chars` is the shared budget for named docs.

## Paperwork

```python
v.categorize(ref, llm='never')   # cues first; model only on ties
v.extract(ref)                   # doctype picks the schema
v.extract(ref, schema='vendor:str, total:float, due_date:str')
v.extract_all(doctype='invoice')
```

`decisive=True` means cues decided without a model.

## Pictures

```python
v.add_image(path, labels=['Superb Fairywren'])   # metadata is the text; pixels stay on disk
v.add_images(dir)
v.label_images(dir, 'aussie-birds')              # anya classifies, the vault files the result
v.tag_image(ref, labels=[...], caption='...')    # write a later model run onto a picture
v.images(); v.search(q, kind='image')
```

`kind='image'`. What is searchable is the filename, folder, dimensions, camera, capture date, GPS,
plus any labels or caption. `label_images` needs [anya](https://github.com/vedicreader/anya).

## Code

```python
v.index_code(dir)                # kosha: AST chunks, symbols, call graph
v.code_search(q); v.symbol(name); v.where_to_add(desc); v.grep(pat, dir)
v.federate(q, dir=dir)           # vault + kosha + ripgrep, fused by rank
```

Once `.kosha/code.db` exists, `context` appends code sections on its own.

## Knobs

Retrieval defaults are litesearch's. Reach for:

- `rerank=True` on `search` / `sections` / `context` when precision matters more than latency
- `v.shelf(name)` to keep two corpora from diluting each other
- `llm=` for how hard `categorize` / `extract` try
- `db.graph_search` for queries that share no words with the answer (costly on ordinary ones)

## Marks and feedback

```python
v.mark_noisy(doc_id, reason='site furniture')
v.mark_pii(doc_id, reason='address book'); v.mark_not_pii(doc_id, reason='my own invoice')
v.pii(doc_id, ner=True)          # read `scanned_ner`, not the count alone
v.suggest_noisy(k=20); v.accept_noisy(k=10)
v.learn()                        # log every ask as feedback
```

Marks live in `doc_marks` (survive re-ingest). `suggest_noisy` is 0.988 AUC. Leave `fit_ranker` / `use_ranker` off unless your corpus says otherwise (`evals/RESULTS.md`).

`connect()` builds the entity graph; `map` / `topic_tree` / `show_topics` read topics. Run `connect()` after a batch of ingests.

## Notes

`stats` / `sources` / `toc` inspect the vault. Ingest is content-addressed. Sanskrit turns on vidyut lemmas and Monier-Williams glosses once.

`forget`, `drop_shelf`, `unwatch` and `pause` are not in `allow`; call them yourself when you mean to.


In [ ]:
#| export
import os
from pyskills import allow
from vishalakshi.core import Vault
from vishalakshi.cli import vault

# Everything a sandboxed agent may call: read the vault, and add to it
READ = ('search sections context related read toc doc document sources stats shelves elsewhere map '
        'doctypes of_type ner categorize extract extract_all ask ask_doc explain doc_context topic_tree show_topics '
        'code_search symbol where_to_add grep federate watches apis assets images').split()
WRITE = ('add add_file add_files add_dir add_tree add_records add_code note grab url web crawl arxiv '
         'pdf youtube github gh_file code connect index_code harvest watch poll run_watch '
         'categorize_all set_meta reshelf shelf route '
         'add_image add_images tag_image label_images').split()

allow(vault, {Vault: READ + WRITE})

`list_pyskills()` reads the module docstring with `ast` (no import): the first paragraph is the summary.


In [ ]:
#| hide
import vishalakshi.skill as _sk
from pyskills.core import list_pyskills, doc

# every name in the allow list is a real Vault attribute, or the registration silently does nothing
for _n in READ + WRITE: assert hasattr(Vault, _n), _n
assert not {'forget', 'drop_shelf', 'unwatch', 'pause'} & set(READ + WRITE)

# the docstring is what an agent is shown; its first paragraph is the one-line description
_first = _sk.__doc__.split('\n\n')[0]
assert 'one vault for everything you have read' in _first, _first
assert 'v.context(q)' in _sk.__doc__ and 'rerank=True' in _sk.__doc__

# registered through the entry point, and discoverable without an import
_ps = list_pyskills()
test_eq(_ps.get('vishalakshi.skill'), _first)

In [ ]:
#| hide
# `vault` is cli's, so the CLI, the MCP server and the pyskill all open the same file
from vishalakshi.cli import vault as _cli_vault
test_eq(vault, _cli_vault)
test_eq(vault.__wrapped__.__defaults__, (None,))

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()